In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Monitorear o cancelar un job

Consulta la lista de tus cargas de trabajo en la [página Workloads](https://quantum.cloud.ibm.com/workloads).

## Ver el estado de un job
Ve a tu [tabla Workloads](https://quantum.cloud.ibm.com/workloads) y revisa la columna Status para saber si un job se completó o falló.

## Ver el uso restante
Ve a tu [tabla Instances](https://quantum.cloud.ibm.com/instances) y selecciona la pestaña asociada al plan del que deseas ver el uso restante. Se muestra el tiempo total utilizado y el tiempo total restante en tu plan.

## Ver métricas sobre el número de jobs y cargas de trabajo enviados
Ve a la [página Analytics](https://quantum.cloud.ibm.com/analytics) para ver el número total de jobs enviados, así como el recuento de cargas de trabajo en lote y en sesión. Ten en cuenta que solo puedes ver la página Analytics para las cuentas que posees o administras.

## Monitorear un job
Usa la instancia del job para verificar su estado o recuperar los resultados llamando al comando apropiado:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | Revisa los resultados del job inmediatamente después de que se complete. Los resultados están disponibles una vez que el job finaliza. Por lo tanto, job.result() es una llamada bloqueante hasta que el job se complete.                               |
| job.job\_id()                 | Devuelve el ID que identifica de forma única ese job. Recuperar los resultados del job en un momento posterior requiere el ID del job. Por ello, se recomienda guardar los IDs de los jobs que puedas querer recuperar más adelante. |
| job.status()                  | Verifica el estado del job.                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | Recupera un job enviado anteriormente. Esta llamada requiere el ID del job.                                                                                                                                      |

<span id="retrieve-later"></span>
## Recuperar resultados de un job en un momento posterior
Llama a `service.job(\<job\_id>)` para recuperar un job que enviaste anteriormente. Si no tienes el ID del job, o si deseas recuperar varios jobs a la vez, incluidos jobs de QPUs (unidades de procesamiento cuántico) retiradas, llama a `service.jobs()` con filtros opcionales. Consulta [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` también devuelve jobs ejecutados desde el paquete obsoleto `qiskit-ibm-provider`. Los jobs enviados por el paquete más antiguo (también obsoleto) `qiskit-ibmq-provider` ya no están disponibles.

### Ejemplo
Este ejemplo devuelve los 10 jobs de runtime más recientes que se ejecutaron en `my_backend`:

In [1]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

<span id="execution-spans"></span>
## View Sampler execution spans

The results of [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2) jobs executed in Qiskit Runtime contain execution timing information in their metadata.
This timing information can be used to place upper and lower timestamp bounds on when particular shots were executed on the QPU.
Shots are grouped into [`ExecutionSpan`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span) objects, each of which indicates a start time, a stop time, and a specification of which shots were collected in the span.

An execution span specifies which data was executed during its window by providing an [`ExecutionSpan.mask`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. This method, given any [Primitive Unified Block (PUB)](/docs/guides/primitive-input-output) index, returns a boolean mask that is `True` for all shots executed during its window. PUBs are indexed by the order in which they were given to the Sampler run call. If, for example, a PUB has shape `(2, 3)` and was run with four shots, then the mask's shape is `(2, 3, 4)`. See the [execution_span](/docs/api/qiskit-ibm-runtime/execution-span) API page for full details.

Example:

To view execution span information, review the metadata of the result returned by `SamplerV2`, which comes in the form of an `ExecutionSpans` object. This object is a list-like container containing instances of subclasses of `ExecutionSpan`, such as `SliceSpan`:

In [6]:
from qiskit.primitives import BitArray

# Get the mask of the 1st PUB for the 0th span.
mask = spans[0].mask(0)

# Decide whether the 0th shot of parameter set (1, 2) occurred in this span.
in_this_span = mask[2, 1, 0]

# Create a new bit array containing only the PUB-1 data collected during this span.
bits = result[0].data.meas
filtered_data = BitArray(bits.array[mask], bits.num_bits)

Execution spans can be filtered to include information pertaining to specific PUBs, selected by their indices:

In [7]:
# take the subset of spans that reference data in PUBs 0 or 2
spans.filter_by_pub([0, 2])

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])

View global information about the collection of execution spans:

In [8]:
print("Number of execution spans:", len(spans))
print("  Start of the first span:", spans.start)
print("     End of the last span:", spans.stop)
print("       Total duration (s):", spans.duration)

Number of execution spans: 1
  Start of the first span: 2025-09-09 16:31:16.320568
     End of the last span: 2025-09-09 16:31:16.865858
       Total duration (s): 0.54529


Extract and inspect a particular span:

In [9]:
spans.sort()
print(" Start of first span:", spans[0].start)
print("   End of first span:", spans[0].stop)
print("#shots in first span:", spans[0].size)

 Start of first span: 2025-09-09 16:31:16.320568
   End of first span: 2025-09-09 16:31:16.865858
#shots in first span: 24


## Cancelar un job
Puedes cancelar un job desde el panel de IBM Quantum Platform, ya sea en la página Workloads o en la página de detalles de una carga de trabajo específica. En la página Workloads, haz clic en el menú de desbordamiento al final de la fila de esa carga de trabajo y selecciona Cancel. Si estás en la página de detalles de una carga de trabajo específica, usa el menú desplegable Actions en la parte superior de la página y selecciona Cancel.

En Qiskit, usa `job.cancel()` para cancelar un job.

<span id="execution-spans"></span>
## Ver los execution spans del Sampler
Los resultados de los jobs de [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) ejecutados en Qiskit Runtime contienen información de temporización de ejecución en sus metadatos.
Esta información de temporización se puede usar para establecer límites de marca de tiempo superior e inferior sobre cuándo se ejecutaron determinadas shots en el QPU.
Las shots se agrupan en objetos [`ExecutionSpan`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span), cada uno de los cuales indica una hora de inicio, una hora de fin y una especificación de qué shots se recopilaron en el span.

Un execution span especifica qué datos se ejecutaron durante su ventana proporcionando un método [`ExecutionSpan.mask`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span#mask). Este método, dado cualquier índice de [Primitive Unified Block (PUB)](/guides/primitive-input-output), devuelve una máscara booleana que es `True` para todas las shots ejecutadas durante su ventana. Los PUBs se indexan por el orden en que se entregaron a la llamada de ejecución del Sampler. Si, por ejemplo, un PUB tiene forma `(2, 3)` y se ejecutó con cuatro shots, entonces la forma de la máscara es `(2, 3, 4)`. Consulta la página de la API [execution_span](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span) para obtener detalles completos.

Ejemplo:
Para ver la información de los execution spans, revisa los metadatos del resultado devuelto por `SamplerV2`, que viene en forma de un objeto `ExecutionSpans`. Este objeto es un contenedor similar a una lista que contiene instancias de subclases de `ExecutionSpan`, como `SliceSpan`: